[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG983_assignment/blob/main/AG983_Assignment_2026.ipynb)

# AG983 | Coursework Assignment 2026

**Textual Analytics for Accounting and Finance**  
University of Strathclyde Business School

---

This notebook serves three purposes: it allocates you to one of four research scenarios based on your student number; it guides you through a structured computational text analysis pipeline; and it generates and submits all required outputs for your written report. Work through every step in order.

**Before you begin — two things to check:**

1. **Make a personal copy.** Go to File → Save a copy in Drive. Work from your own copy — do not edit the shared original.

2. **Set your runtime type.** If you plan to use **FinBERT** or **DistilBERT** as your sentiment model (Step 6), you should switch to a GPU runtime *before running Step 0* — changing runtime later will reset your session and you will have to start again. Go to **Runtime → Change runtime type → T4 GPU**. This is available on the free tier of Colab and does not require a paid account. If you plan to use one of the four classical models (LM dictionary, Harvard IV, Naive Bayes, or Logistic Regression), no GPU is needed and you can leave the runtime as the default CPU.

In [ ]:
#@title Step 0 -- Clone the AG983 repository (run this first)

import os
import subprocess

REPO_URL = "https://github.com/iamjamesbowden/AG983_assignment.git"
REPO_DIR = "AG983_assignment"

if not os.path.exists(REPO_DIR):
    print("Cloning AG983 repository...")
    result = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    if result.returncode == 0:
        print("Repository cloned successfully.")
    else:
        print("Clone failed:\n", result.stderr)
else:
    print("Repository already present — pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True)
    print("Done.")

# Verify data files are present
data_root = os.path.join(REPO_DIR, "data")
if os.path.isdir(data_root):
    print("Data directory found:", data_root)
else:
    print("WARNING: data directory not found at", data_root)
    print("Contact the module team if this error persists.")


In [ ]:
#@title Configuration -- set APPS_SCRIPT_URL before running

APPS_SCRIPT_URL = "https://script.google.com/macros/s/AKfycbwPznLTlJz_KYSulVB_6kx_5yY8q_F2nK07xVq987N25zbyRKxS7djTQOMUGWS5PIQT/exec"

# Paths — set after Step 0 clones the repo into AG983_assignment/
REPO_DIR          = "AG983_assignment"
LM_DICT_PATH      = f"{REPO_DIR}/data/lm_dictionary.csv"
HARVARD_DICT_PATH = f"{REPO_DIR}/data/harvard_iv_dictionary.csv"
SEED_LABELS_PATH  = f"{REPO_DIR}/data/nb_seed_labels.csv"


In [ ]:
#@title Step 1 -- Install and import dependencies

!pip install requests pandas numpy matplotlib seaborn scikit-learn nltk transformers torch sentencepiece -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, os, json, time, warnings, requests, random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)

warnings.filterwarnings('ignore')
print("Dependencies installed and imported successfully.")


## Scenario Allocation

Each student is assigned to one of four research scenarios based on their student number. The scenario determines which corpus and research question you will work with throughout the assignment. You must not change your allocated scenario.

**What to do:**
1. Run the cell below (click the ▶ play button to the left of it, or press Shift+Enter).
2. An input box will appear **at the bottom of the cell output area** — scroll down if you cannot see it.
3. Type your student number exactly and press Enter.
4. Make a note of your assigned scenario. All subsequent steps depend on it.


In [ ]:
#@title Step 2 -- Enter your student number to receive your scenario allocation

SCENARIOS = {
    "A": {
        "id": "A",
        "title": "Cybersecurity Risk Disclosure (2019–2024)",
        "question": (
            "How has the language used to disclose cybersecurity risk in annual "
            "report filings evolved across financial services, healthcare, technology, "
            "and retail firms between 2019 and 2024, and did the SEC's December 2023 "
            "mandatory cyber disclosure rules produce measurable shifts in disclosure "
            "sentiment or thematic content?"
        ),
        "firms": 49, "filings": "~294",
        "categories": ["financial_services", "healthcare", "technology", "retail_hospitality"],
        "years": [2019, 2020, 2021, 2022, 2023, 2024],
        "corpus_path": "AG983_assignment/data/scenario_a/corpus.csv"
    },
    "B": {
        "id": "B",
        "title": "Consumer ESG and Greenwashing Risk (2019–2024)",
        "question": (
            "How have consumer-facing firms disclosed sustainability commitments "
            "and associated risks between 2019 and 2024, and is there evidence that "
            "disclosure language shifted in response to greenwashing litigation, "
            "SEC ESG proposals, and FTC Green Guides scrutiny?"
        ),
        "firms": 52, "filings": "~312",
        "categories": ["consumer_staples", "consumer_discretionary", "food_beverage", "retail"],
        "years": [2019, 2020, 2021, 2022, 2023, 2024],
        "corpus_path": "AG983_assignment/data/scenario_b/corpus.csv"
    },
    "C": {
        "id": "C",
        "title": "Pharmaceutical Liability and Opioid Litigation (2015–2024)",
        "question": (
            "How did the textual characteristics of risk and litigation disclosures "
            "evolve across pharmaceutical manufacturers, wholesale distributors, and "
            "pharmacy chains between 2015 and 2024 as the opioid litigation arc "
            "progressed from early lawsuits through DOJ investigations to "
            "multi-billion dollar settlements?"
        ),
        "firms": 37, "filings": "~370",
        "categories": ["defendant", "distributor", "pharmacy_chain", "big_pharma", "specialty_pharma"],
        "years": [2015,2016,2017,2018,2019,2020,2021,2022,2023,2024],
        "corpus_path": "AG983_assignment/data/scenario_c/corpus.csv"
    },
    "D": {
        "id": "D",
        "title": "Real Estate and Interest Rate Risk (2018–2024)",
        "question": (
            "How did REIT and commercial real estate firms disclose interest rate "
            "risk and refinancing exposure between 2018 and 2024, and do textual "
            "signals in pre-hike filings (2018–2021) differ systematically from "
            "those in the rate-hike period (2022–2024)?"
        ),
        "firms": 47, "filings": "~329",
        "categories": ["industrial_reit","office_reit","retail_reit","residential_reit","specialty_reit","re_services"],
        "years": [2018, 2019, 2020, 2021, 2022, 2023, 2024],
        "corpus_path": "AG983_assignment/data/scenario_d/corpus.csv"
    },
}

STUDENT_ID = input("Enter your student number: ").strip()

# Deterministic allocation by student number
scenario_keys = list(SCENARIOS.keys())
try:
    _digits = int(''.join(filter(str.isdigit, STUDENT_ID)))
except ValueError:
    _digits = 0
SCENARIO = SCENARIOS[scenario_keys[_digits % 4]]

# Derive reproducible personal seed from student number
STUDENT_SEED = _digits % (2**31 - 1) if _digits > 0 else 42

print(f"\nStudent ID   : {STUDENT_ID}")
print(f"Scenario     : {SCENARIO['id']} — {SCENARIO['title']}")
print(f"Research Q   : {SCENARIO['question']}")
print(f"Corpus       : {SCENARIO['firms']} firms, approx. {SCENARIO['filings']} filings")
print(f"Categories   : {', '.join(SCENARIO['categories'])}")
print(f"Years        : {SCENARIO['years'][0]}–{SCENARIO['years'][-1]}")
print(f"Personal seed: {STUDENT_SEED} (used for reproducible random sampling)")

CORPUS_PATH = SCENARIO["corpus_path"]
LM_DICT_PATH = "AG983_assignment/data/lm_dictionary.csv"
HARVARD_DICT_PATH = "AG983_assignment/data/harvard_iv_dictionary.csv"
SEED_LABELS_PATH = "AG983_assignment/data/nb_seed_labels.csv"


In [ ]:
#@title Assignment workflow

def _draw_workflow_diagram():
    """Render the assignment step workflow as a horizontal pipeline."""
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt

    stages = [
        ("3",     "Load\nCorpus"),
        ("4\u20135",  "Pre-\nprocessing"),
        ("6\u20136e", "Sentiment\nAnalysis"),
        ("7\u20139",  "Secondary\nMetric"),
        ("10",    "Visuali-\nsation"),
        ("11",    "Manual\nValidation"),
        ("12\u201313","Submit &\nSummary"),
    ]
    colours = ["#4f46e5", "#0ea5e9", "#10b981", "#f59e0b", "#ef4444", "#8b5cf6", "#06b6d4"]

    fig, ax = plt.subplots(figsize=(13, 2.4))
    ax.set_xlim(0, len(stages))
    ax.set_ylim(0, 1)
    ax.axis("off")
    bw, bh, by = 0.82, 0.56, 0.22

    for i, ((code, label), col) in enumerate(zip(stages, colours)):
        cx = i + 0.5
        rect = mpatches.FancyBboxPatch(
            (cx - bw / 2, by), bw, bh,
            boxstyle="round,pad=0.04", linewidth=1.2,
            edgecolor="white", facecolor=col, zorder=3)
        ax.add_patch(rect)
        ax.text(cx, by + bh / 2 + 0.08, f"Step {code}",
                ha="center", va="center",
                fontsize=7.5, fontweight="bold", color="white", zorder=4)
        ax.text(cx, by + bh / 2 - 0.10, label,
                ha="center", va="center",
                fontsize=6.5, color="white", zorder=4, linespacing=1.3)
        if i < len(stages) - 1:
            ax.annotate("", xy=(i + 1 + 0.5 - bw / 2 - 0.02, by + bh / 2),
                        xytext=(cx + bw / 2 + 0.02, by + bh / 2),
                        arrowprops=dict(arrowstyle="->", color="#64748b", lw=1.4),
                        zorder=2)

    ax.set_title("Assignment workflow \u2014 complete steps in order",
                 fontsize=9, color="#475569", pad=6)
    plt.tight_layout()
    plt.show()

_draw_workflow_diagram()

In [ ]:
#@title Step 3 -- Load the corpus

# The corpus contains two sections for each firm and year:
#
#   item_1a  --  Risk Factors
#   item_7   --  Management's Discussion and Analysis (MD&A)
#
# Use the dropdown below to select which section(s) you will analyse.
# Your choice has direct consequences for which linguistic phenomena are
# captured and must be justified in your report with reference to the
# literature and your scenario's research question.

import ipywidgets as widgets
from IPython.display import display

# CORPUS_PATH was set in Step 2 from the scenario allocation
CORPUS_PATH = SCENARIO["corpus_path"]

try:
    _raw_corpus = pd.read_csv(CORPUS_PATH)
    print(f"Corpus loaded from: {CORPUS_PATH}")
    print(f"Raw shape: {_raw_corpus.shape[0]} rows x {_raw_corpus.shape[1]} columns")
    print(f"Columns:   {list(_raw_corpus.columns)}")
    print(f"Firms:     {_raw_corpus['firm'].nunique()}")
    print(f"Years:     {sorted(_raw_corpus['year'].unique())}")
    print(f"Sections:  {sorted(_raw_corpus['section'].unique())}")
    print()
except FileNotFoundError:
    print(f"ERROR: corpus not found at {CORPUS_PATH}.")
    print("Run `git pull origin main` to ensure the data files are present.")
    _raw_corpus = None

w_section = widgets.Dropdown(
    options=[
        "Both sections combined (full narrative)",
        "item_1a \u2014 Risk Factors only",
        "item_7 \u2014 MD\u0026A only",
    ],
    description="Corpus section:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="520px"),
)

_section_out = widgets.Output()

def _apply_section(_change=None):
    global corpus
    if _raw_corpus is None:
        return
    _section_out.clear_output(wait=True)
    with _section_out:
        if w_section.value == "item_1a \u2014 Risk Factors only":
            corpus = _raw_corpus[_raw_corpus["section"] == "item_1a"].copy()
        elif w_section.value == "item_7 \u2014 MD\u0026A only":
            corpus = _raw_corpus[_raw_corpus["section"] == "item_7"].copy()
        else:
            # Combine both sections into one row per firm-year
            _agg = {"text": " ".join}
            if "word_count" in _raw_corpus.columns:
                _agg["word_count"] = "sum"
            corpus = (_raw_corpus
                      .groupby(["firm", "category", "year"], as_index=False)
                      .agg(_agg))
            corpus["section"] = "combined"

        print(f"Section selection applied: {w_section.value}")
        print(f"Working corpus: {corpus.shape[0]} rows x {corpus.shape[1]} columns")
        print(f"Firms: {corpus['firm'].nunique()} | "
              f"Years: {sorted(corpus['year'].unique())}")
        print()
        print("This choice is recorded and submitted with your other")
        print("methodological decisions when you run Step 12.")

w_section.observe(_apply_section, names="value")

print("Select the corpus section you will analyse, then continue to Step 4.")
display(w_section, _section_out)
_apply_section()

## Pre-processing

You must select a cleaning pipeline that prepares the raw 10-K text for downstream analysis and justify every decision in your report with reference to relevant literature, including lecture slides and supporting class activities, core reading and external reading. Five decisions are required, each with direct consequences for the validity of your results. Choices made without theoretical justification will be penalised in the marking scheme.

| Decision | What the choice affects |
|---|---|
| Case folding | Whether all tokens are lowercased prior to analysis; affects dictionary matching, topic model vocabulary, and the distinction between proper nouns and common words |
| Stop-word list | Which words are excluded from the token set prior to analysis |
| Stemming vs. lemmatisation | How inflected word forms are handled; affects vocabulary size and the mapping of related terms |
| Number handling | Whether numeric tokens are retained in, removed from, or replaced within the token set |
| TF-IDF weighting | Whether term frequencies are adjusted by their rarity across the corpus before scoring |

**What to do:**
1. **Run Step 4.** Five dropdown menus will appear. Use them to record your pre-processing choices — the selections are saved automatically as you change them.
2. **Run Step 5.** This applies your choices to the corpus. The first 30 tokens of a sample document are printed so you can verify the pipeline is working as expected.

In [ ]:
#@title Step 4 -- Record your pre-processing decisions

!pip install ipywidgets -q

import ipywidgets as widgets
from IPython.display import display

w_case_folding = widgets.Dropdown(
    options=["Yes — lowercase all tokens", "No — preserve original case"],
    description="Case folding:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_stopwords = widgets.Dropdown(
    options=["Standard NLTK stopwords",
             "Finance-adjusted (modal verbs and negation retained)"],
    description="Stop-word list:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_normalisation = widgets.Dropdown(
    options=["Lemmatisation (WordNetLemmatizer)", "Stemming (PorterStemmer)", "None"],
    description="Normalisation:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_numbers = widgets.Dropdown(
    options=["Remove all numeric tokens", "Retain as-is",
             "Replace with placeholder <NUM>"],
    description="Number handling:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_tfidf = widgets.Dropdown(
    options=["Yes -- apply TF-IDF weighting", "No -- use raw term counts"],
    description="TF-IDF weighting:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)

# Live confirmation panel -- updates immediately when any dropdown changes
_out = widgets.Output()

def _show_selections(_change=None):
    _out.clear_output(wait=True)
    with _out:
        print("Choices recorded:")
        print(f"  Case folding:     {w_case_folding.value}")
        print(f"  Stop-word list:   {w_stopwords.value}")
        print(f"  Normalisation:    {w_normalisation.value}")
        print(f"  Number handling:  {w_numbers.value}")
        print(f"  TF-IDF weighting: {w_tfidf.value}")
        print()
        print("These values are stored in the Python session. They take effect")
        print("once you implement preprocess() in Step 5, and are submitted to")
        print("the module log when you run Step 13.")

for _w in [w_case_folding, w_stopwords, w_normalisation, w_numbers, w_tfidf]:
    _w.observe(_show_selections, names="value")

display(w_case_folding, w_stopwords, w_normalisation, w_numbers, w_tfidf, _out)
_show_selections()  # show initial state immediately

In [ ]:
#@title Step 5 -- Apply pre-processing pipeline

import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Modal verbs and negation retained under the finance-adjusted stop-word list
_FINANCE_RETAIN = {
    "no", "not", "nor", "neither", "never", "none",
    "can", "cannot", "could", "may", "might",
    "shall", "should", "will", "would", "must"
}

# Regex for tokens that are purely numeric (digits, commas, decimals, signs, currency symbols)
_NUM_RE = re.compile(r'^[\d,.\-\+%$£€]+$')


def preprocess(text: str) -> list:
    """
    Clean and tokenise a single document string using the choices
    selected in Step 4.

    Pipeline (in order):
      1. Tokenise with NLTK word_tokenize.
      2. Case folding       -- controlled by w_case_folding.value.
      3. Number handling    -- controlled by w_numbers.value.
      4. Remove punctuation-only tokens.
      5. Stop-word removal  -- controlled by w_stopwords.value.
      6. Normalisation      -- controlled by w_normalisation.value.

    TF-IDF weighting (w_tfidf.value) is a corpus-level operation applied
    in Steps 6a and 6b; it is not part of single-document tokenisation.
    """
    # 1. Tokenise
    tokens = word_tokenize(str(text))

    # 2. Case folding
    if w_case_folding.value == "Yes — lowercase all tokens":
        tokens = [t.lower() for t in tokens]

    # 3. Number handling
    if w_numbers.value == "Remove all numeric tokens":
        tokens = [t for t in tokens if not _NUM_RE.match(t)]
    elif w_numbers.value == "Replace with placeholder <NUM>":
        tokens = ["<NUM>" if _NUM_RE.match(t) else t for t in tokens]
    # "Retain as-is" -- no change

    # 4. Remove punctuation-only tokens (preserve <NUM> placeholder)
    tokens = [t for t in tokens if t == "<NUM>" or re.search(r'[a-zA-Z]', t)]

    # 5. Stop-word removal (compare lowercase regardless of case folding choice,
    #    so common words are still removed even when case is preserved)
    _stop = set(stopwords.words("english"))
    if w_stopwords.value == "Finance-adjusted (modal verbs and negation retained)":
        _stop = _stop - _FINANCE_RETAIN
    tokens = [t for t in tokens if t.lower() not in _stop]

    # 6. Normalisation
    if w_normalisation.value == "Lemmatisation (WordNetLemmatizer)":
        _lem = WordNetLemmatizer()
        tokens = [_lem.lemmatize(t) for t in tokens]
    elif w_normalisation.value == "Stemming (PorterStemmer)":
        _stem = PorterStemmer()
        tokens = [_stem.stem(t) for t in tokens]
    # "None" -- no change

    return tokens


# Test on the first row of the corpus
try:
    sample_text = corpus["text"].iloc[0] if "text" in corpus.columns else ""
    tokens = preprocess(sample_text)
    print("Pre-processing applied successfully.")
    print(f"  Case folding:     {w_case_folding.value}")
    print(f"  Stop-word list:   {w_stopwords.value}")
    print(f"  Normalisation:    {w_normalisation.value}")
    print(f"  Number handling:  {w_numbers.value}")
    print(f"  TF-IDF weighting: {w_tfidf.value} (applied at corpus level in Steps 6a/6b)")
    print(f"\nSample document: {len(tokens)} tokens after pre-processing.")
    print(f"First 30 tokens:  {tokens[:30]}")
except NameError:
    print("corpus is not yet defined -- run Step 3 first, then re-run this cell.")

## Sentiment Analysis

You must apply one sentiment model to the corpus and justify your choice in the report on at least three criteria: transparency of the underlying word list or model weights; domain specificity relative to the text being analysed; and computational requirements relative to corpus size. Six approaches are available.

The **Loughran-McDonald (LM) dictionary** and the **Harvard General Inquirer (Harvard IV)** are rule-based methods that match words against pre-compiled sentiment word lists. They are fast, interpretable, and widely used in accounting and finance research, but treat each word in isolation and cannot account for context.

The **Naive Bayes** and **Logistic Regression** classifiers are machine learning methods trained on 286 human-labelled seed sentences drawn from financial filings. They learn statistical patterns in the training data rather than relying on a hand-crafted word list, and are evaluated by 5-fold stratified cross-validation.

**FinBERT** (yiyanghkust/finbert-tone) is a transformer-based model pre-trained on financial text. Unlike bag-of-words methods, it encodes full sentence context and handles negation and modifier phrases that dictionary methods miss. It is domain-specific but computationally heavier. **If you choose FinBERT, you must use a GPU runtime.** Switch before starting: Runtime → Change runtime type → T4 GPU. The T4 GPU is available on the free tier — no payment required. Be aware that GPU availability on the free tier is not guaranteed; if Colab reports no GPU is available, try again at a less busy time or switch to one of the classical models.

**DistilBERT** (distilbert-base-uncased-finetuned-sst-2-english) is a distilled, general-purpose transformer approximately 60% faster than BERT-base while retaining ~97% of its performance on sentiment tasks. A GPU is recommended for DistilBERT but not strictly required — on CPU it will run in 20–40 minutes for a corpus of this size. The same T4 GPU runtime applies (free tier).

All six methods produce a continuous score in [−1, +1]. The thresholds you set in Step 6 determine how this score is converted into a categorical label.

> **GPU users:** if you have not already switched to a T4 GPU runtime, do so now before proceeding — Runtime → Change runtime type → T4 GPU — then re-run Steps 0–5 before continuing here.

In [ ]:
#@title Step 6 -- Select your sentiment model and set classification thresholds

import ipywidgets as widgets
from IPython.display import display

w_sentiment = widgets.Dropdown(
    options=[
        "lm_dictionary",
        "harvard_iv",
        "naive_bayes",
        "logistic_regression",
        "finbert",
        "distilbert",
    ],
    description="Sentiment model:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="520px")
)

w_pos_threshold = widgets.FloatSlider(
    value=0.10, min=0.0, max=0.5, step=0.01,
    description="Positive threshold (≥):",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="520px")
)
w_neg_threshold = widgets.FloatSlider(
    value=-0.10, min=-0.5, max=0.0, step=0.01,
    description="Negative threshold (≤):",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="520px")
)

_model_notes = {
    "lm_dictionary":      "Dictionary — no GPU required",
    "harvard_iv":         "Dictionary — no GPU required",
    "naive_bayes":        "ML classifier — no GPU required",
    "logistic_regression":"ML classifier — no GPU required",
    "finbert":            "Transformer — GPU strongly recommended (Runtime → T4 GPU)",
    "distilbert":         "Transformer — GPU recommended for large corpora",
}

def _on_model_change(change):
    note = _model_notes.get(change['new'], '')
    print(f"\rSelected: {change['new']}  |  {note}         ", end='')

w_sentiment.observe(_on_model_change, names='value')

print("Set your choices below, then proceed to the appropriate Step 6 sub-cell.")
print("  • lm_dictionary / harvard_iv      → run Step 6a")
print("  • naive_bayes / logistic_regression → run Step 6b")
print("  • finbert                          → run Step 6c")
print("  • distilbert                       → run Step 6d")
print("  All students run Step 6e (sentiment score table).\n")
display(w_sentiment, w_pos_threshold, w_neg_threshold)


In [ ]:
#@title Step 6a -- Dictionary sentiment (lm_dictionary or harvard_iv)

# Run this cell if w_sentiment.value is "lm_dictionary" or "harvard_iv".
# Skip it if you chose a classifier approach; run Step 6b instead.

if w_sentiment.value in ("lm_dictionary", "harvard_iv"):

    dict_path = LM_DICT_PATH if w_sentiment.value == "lm_dictionary" else HARVARD_DICT_PATH

    try:
        d = pd.read_csv(dict_path)
        positive_words    = set(d[d["Positive"]    != 0]["Word"].str.lower())
        negative_words    = set(d[d["Negative"]    != 0]["Word"].str.lower())
        uncertainty_words = (set(d[d["Uncertainty"] != 0]["Word"].str.lower())
                             if "Uncertainty" in d.columns else set())
        unc_note = f", {len(uncertainty_words)} uncertainty" if uncertainty_words else ""
        print(f"{w_sentiment.value} loaded: "
              f"{len(positive_words)} positive, {len(negative_words)} negative{unc_note}.")
    except FileNotFoundError:
        print(f"ERROR: dictionary not found at {dict_path}.")
        print("Run `git pull origin main` to ensure data files are present.")
        positive_words, negative_words, uncertainty_words = set(), set(), set()


    def dict_sentiment(text, pos_words, neg_words, unc_words):
        """
        Compute dictionary sentiment scores for a single document.
        Returns a dict with pos_count, neg_count, net_sentiment,
        sentiment_score (continuous, -1 to +1), and unc_count.
        """
        try:
            _tokens = preprocess(str(text))
            tokens = [t.lower() for t in _tokens] if _tokens else \
                     [t.lower() for t in str(text).split()]
        except Exception:
            tokens = [t.lower() for t in str(text).split()]

        pos   = sum(1 for t in tokens if t in pos_words)
        neg   = sum(1 for t in tokens if t in neg_words)
        unc   = sum(1 for t in tokens if t in unc_words)
        denom = pos + neg
        score = (pos - neg) / denom if denom > 0 else np.nan
        return {"pos_count": pos, "neg_count": neg,
                "net_sentiment": pos - neg, "sentiment_score": score,
                "unc_count": unc}


    # Score the full corpus
    if "text" in corpus.columns and positive_words:
        results = corpus["text"].apply(
            lambda t: dict_sentiment(t, positive_words, negative_words, uncertainty_words)
        )
        corpus["dict_pos"]   = results.apply(lambda r: r["pos_count"])
        corpus["dict_neg"]   = results.apply(lambda r: r["neg_count"])
        corpus["dict_net"]   = results.apply(lambda r: r["net_sentiment"])
        corpus["dict_score"] = results.apply(lambda r: r["sentiment_score"])
        if uncertainty_words:
            corpus["dict_unc"] = results.apply(lambda r: r["unc_count"])
        print("Dictionary sentiment scores computed.")
        display(corpus[["dict_pos", "dict_neg", "dict_score"]].describe())


    # -------------------------------------------------------------------------
    # Evaluate dictionary accuracy against seed labels
    # -------------------------------------------------------------------------
    # The thresholds set in Step 6 are used here. Adjusting them in Step 6
    # will automatically update the confusion matrix below.
    # -------------------------------------------------------------------------

    try:
        _seed = pd.read_csv(SEED_LABELS_PATH)
        _seed_scores = _seed["sentence"].apply(
            lambda t: dict_sentiment(t, positive_words, negative_words, uncertainty_words)
        )
        _seed["dict_score"] = _seed_scores.apply(lambda r: r["sentiment_score"])

        print("\n" + "-" * 60)
        print("Sentiment score distribution across seed sentences:")
        print("(Use this to inform your threshold choices in Step 6.)")
        print("-" * 60)
        display(_seed["dict_score"].describe().to_frame().T.round(4))

        _eval_out = widgets.Output()

        def _run_evaluation(_change=None):
            _eval_out.clear_output(wait=True)
            with _eval_out:
                pos_t = w_pos_threshold.value
                neg_t = w_neg_threshold.value
                label_order = ["Positive", "Neutral", "Negative"]

                _seed["predicted_label"] = _seed["dict_score"].apply(_score_to_label)

                print(f"Thresholds: Positive \u2265 {pos_t:+.2f} | "
                      f"Negative \u2264 {neg_t:+.2f} | "
                      f"Neutral: {neg_t:+.2f} to {pos_t:+.2f}")
                print(f"n = {len(_seed)} seed sentences\n")

                print("Predicted label distribution:")
                display(_seed["predicted_label"].value_counts().to_frame())

                # Classification metrics (without support column)
                _rep = classification_report(
                    _seed["label"], _seed["predicted_label"],
                    labels=label_order, output_dict=True, zero_division=0
                )
                _rep_df = (pd.DataFrame(_rep).T
                           .loc[label_order, ["precision", "recall", "f1-score"]]
                           .round(3))
                _rep_df.columns = ["Precision", "Recall", "F1"]
                print("\nClassification metrics (dictionary vs. human seed labels):")
                display(_rep_df)

                # Confusion matrix -- displayed and printed for easy copy-paste
                cm = confusion_matrix(
                    _seed["label"], _seed["predicted_label"], labels=label_order
                )
                cm_df = pd.DataFrame(cm, index=label_order, columns=label_order)
                cm_df.index.name   = "Actual"
                cm_df.columns.name = "Predicted"
                print("\nConfusion matrix:")
                display(cm_df)
                print(cm_df.to_string())

        # Connect to the shared threshold widgets defined in Step 6
        w_pos_threshold.observe(_run_evaluation, names="value")
        w_neg_threshold.observe(_run_evaluation, names="value")

        print("\nThe confusion matrix updates automatically when you adjust")
        print("the thresholds in Step 6 above.")
        display(_eval_out)
        _run_evaluation()

    except FileNotFoundError:
        print(f"\nNote: seed labels not found at {SEED_LABELS_PATH} -- evaluation skipped.")

else:
    print(f"Skipping Step 6a: sentiment model is '{w_sentiment.value}'. Run Step 6b instead.")

In [ ]:
#@title Step 6b -- ML classifier sentiment (naive_bayes or logistic_regression)

# Run this cell if w_sentiment.value is 'naive_bayes' or 'logistic_regression'.

if w_sentiment.value in ('naive_bayes', 'logistic_regression'):

    seeds = pd.read_csv(SEED_LABELS_PATH)
    seeds = seeds.dropna(subset=['sentence','label'])
    _label_map = {'Positive': 1, 'Neutral': 0, 'Negative': -1}
    seeds['label_int'] = seeds['label'].map(_label_map)
    y_all = seeds['label_int'].values

    vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
    X_all = vec.fit_transform(seeds['sentence'])

    if w_sentiment.value == 'naive_bayes':
        from sklearn.naive_bayes import ComplementNB
        clf_class = ComplementNB
        clf_kwargs = {}
    else:
        clf_class = LogisticRegression
        clf_kwargs = {'max_iter': 1000, 'random_state': STUDENT_SEED, 'C': 1.0}

    # 5-fold stratified cross-validation on seed sentences
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=STUDENT_SEED)
    cv_preds = np.zeros(len(y_all), dtype=int)
    for train_idx, val_idx in skf.split(X_all, y_all):
        clf_cv = clf_class(**clf_kwargs).fit(X_all[train_idx], y_all[train_idx])
        cv_preds[val_idx] = clf_cv.predict(X_all[val_idx])

    labels_str = ['Negative', 'Neutral', 'Positive']
    p, r, f, _ = precision_recall_fscore_support(y_all, cv_preds, labels=[-1, 0, 1], zero_division=0)
    cm = confusion_matrix(y_all, cv_preds)

    print(f'\n=== {w_sentiment.value} — 5-fold CV on {len(seeds)} seed sentences ===')
    print(f'{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}')
    for i, lbl in enumerate(labels_str):
        print(f'{lbl:<12} {p[i]:>10.3f} {r[i]:>10.3f} {f[i]:>10.3f}')
    print('\nConfusion matrix (rows=actual, cols=predicted):')
    print(f"{'':12}" + ''.join(f'{l:>12}' for l in labels_str))
    for i, row in enumerate(cm):
        print(f'{labels_str[i]:<12}' + ''.join(f'{v:>12}' for v in row))

    # Train final model on all seed sentences
    clf_final = clf_class(**clf_kwargs).fit(X_all, y_all)

    # Score the corpus
    X_corpus = vec.transform(corpus['text'])
    proba = clf_final.predict_proba(X_corpus)
    corpus['ml_score'] = proba[:, 2] - proba[:, 0]  # P(Positive) - P(Negative)
    print('\nCorpus scored. Proceed to Step 6e.')
else:
    print('Skipping Step 6b — you selected:', w_sentiment.value)


In [ ]:
#@title Step 6c -- FinBERT sentiment (domain-specific transformer)

# Run this cell ONLY if w_sentiment.value == 'finbert'.
# Requires GPU: Runtime → Change runtime type → T4 GPU

if w_sentiment.value == 'finbert':
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch.nn.functional as F

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cpu':
        print('WARNING: No GPU detected. FinBERT on CPU may take 30-60 minutes.')
        print('Recommended: Runtime → Change runtime type → T4 GPU')
    else:
        print(f'GPU detected: {torch.cuda.get_device_name(0)}')

    MODEL_NAME = 'yiyanghkust/finbert-tone'
    print(f'Loading {MODEL_NAME}...')
    _tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    _model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
    _model.eval()
    _id2label = _model.config.id2label  # e.g. {0:'Positive', 1:'Negative', 2:'Neutral'}
    _pos_idx = next(k for k, v in _id2label.items() if v.lower() == 'positive')
    _neg_idx = next(k for k, v in _id2label.items() if v.lower() == 'negative')

    def _score_finbert(text, batch_size=32, max_sents=300):
        sents = sent_tokenize(str(text))[:max_sents]
        if not sents:
            return 0.0
        scores = []
        for i in range(0, len(sents), batch_size):
            batch = sents[i:i+batch_size]
            enc = _tokenizer(batch, return_tensors='pt', truncation=True,
                             max_length=128, padding=True).to(device)
            with torch.no_grad():
                probs = F.softmax(_model(**enc).logits, dim=-1).cpu().numpy()
            scores.extend(float(p[_pos_idx] - p[_neg_idx]) for p in probs)
        return float(np.mean(scores))

    print('Scoring corpus with FinBERT (sentence-level batching)...')
    corpus['ml_score'] = corpus['text'].apply(_score_finbert)
    print(f'Done. Scored {len(corpus)} documents.')

    # Evaluate on seed sentences
    seeds = pd.read_csv(SEED_LABELS_PATH).dropna(subset=['sentence','label'])
    _label_map = {'Positive': 1, 'Neutral': 0, 'Negative': -1}
    seeds['label_int'] = seeds['label'].map(_label_map)
    seed_scores = seeds['sentence'].apply(_score_finbert)
    pos_t, neg_t = w_pos_threshold.value, w_neg_threshold.value
    pred_labels = seed_scores.apply(lambda s: 1 if s >= pos_t else (-1 if s <= neg_t else 0))
    true_labels = seeds['label_int']
    p, r, f, _ = precision_recall_fscore_support(true_labels, pred_labels, labels=[-1,0,1], zero_division=0)
    cm = confusion_matrix(true_labels, pred_labels, labels=[-1,0,1])
    lbl_names = ['Negative','Neutral','Positive']
    print(f'\n=== FinBERT evaluation on {len(seeds)} seed sentences ===')
    print(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    for i, nm in enumerate(lbl_names):
        print(f'{nm:<12} {p[i]:>10.3f} {r[i]:>10.3f} {f[i]:>10.3f}')
    print('\nConfusion matrix (rows=actual, cols=predicted):')
    print(f"{'':12}" + ''.join(f'{l:>12}' for l in lbl_names))
    for i, row in enumerate(cm):
        print(f'{lbl_names[i]:<12}' + ''.join(f'{v:>12}' for v in row))
    print('\nProceed to Step 6e.')
else:
    print('Skipping Step 6c — you selected:', w_sentiment.value)


In [ ]:
#@title Step 6d -- DistilBERT sentiment (general-purpose transformer)

# Run this cell ONLY if w_sentiment.value == 'distilbert'.
# GPU recommended but not required for smaller corpora.

if w_sentiment.value == 'distilbert':
    import torch
    from transformers import pipeline as hf_pipeline, logging as hf_logging
    hf_logging.set_verbosity_error()  # suppress sequential-pipeline performance warning

    device = 0 if torch.cuda.is_available() else -1
    if device == -1:
        print('No GPU detected — running DistilBERT on CPU (may be slow for large corpora).')
    else:
        print(f'GPU: {torch.cuda.get_device_name(0)}')

    _pipe = hf_pipeline(
        'sentiment-analysis',
        model='distilbert-base-uncased-finetuned-sst-2-english',
        device=device, truncation=True, max_length=512
    )

    def _score_distilbert(text, batch_size=32, max_sents=300):
        sents = sent_tokenize(str(text))[:max_sents]
        if not sents:
            return 0.0
        scores = []
        for i in range(0, len(sents), batch_size):
            batch = sents[i:i+batch_size]
            results = _pipe(batch, batch_size=batch_size)
            for r in results:
                s = r['score'] if r['label'] == 'POSITIVE' else -r['score']
                scores.append(s)
        return float(np.mean(scores))

    print('Scoring corpus with DistilBERT (sentence-level batching)...')
    corpus['ml_score'] = corpus['text'].apply(_score_distilbert)
    print(f'Done. Scored {len(corpus)} documents.')

    # Evaluate on seed sentences
    seeds = pd.read_csv(SEED_LABELS_PATH).dropna(subset=['sentence','label'])
    _label_map = {'Positive': 1, 'Neutral': 0, 'Negative': -1}
    seeds['label_int'] = seeds['label'].map(_label_map)
    seed_scores = seeds['sentence'].apply(_score_distilbert)
    pos_t, neg_t = w_pos_threshold.value, w_neg_threshold.value
    pred_labels = seed_scores.apply(lambda s: 1 if s >= pos_t else (-1 if s <= neg_t else 0))
    true_labels = seeds['label_int']
    p, r, f, _ = precision_recall_fscore_support(true_labels, pred_labels, labels=[-1,0,1], zero_division=0)
    cm = confusion_matrix(true_labels, pred_labels, labels=[-1,0,1])
    lbl_names = ['Negative','Neutral','Positive']
    print(f'\n=== DistilBERT evaluation on {len(seeds)} seed sentences ===')
    print(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    for i, nm in enumerate(lbl_names):
        print(f'{nm:<12} {p[i]:>10.3f} {r[i]:>10.3f} {f[i]:>10.3f}')
    print('\nConfusion matrix (rows=actual, cols=predicted):')
    print(f"{'':12}" + ''.join(f'{l:>12}' for l in lbl_names))
    for i, row in enumerate(cm):
        print(f'{lbl_names[i]:<12}' + ''.join(f'{v:>12}' for v in row))
    print('\nProceed to Step 6e.')
else:
    print('Skipping Step 6d — you selected:', w_sentiment.value)


In [ ]:
#@title Step 6e -- Sentiment score table

# This cell pivots your sentiment scores into a company x year matrix.
# Run it after whichever Step 6 sub-cell you ran (6a, 6b, 6c, or 6d).

_df = corpus

score_col = ("dict_score" if "dict_score" in _df.columns else
             "ml_score"   if "ml_score"   in _df.columns else None)

if score_col is None:
    print("No sentiment scores found in the corpus. Run Step 6a or Step 6b first.")
else:
    pivot = (_df
             .pivot_table(index=["firm", "category"],
                          columns="year",
                          values=score_col,
                          aggfunc="mean")
             .round(4))

    pivot = pivot.reset_index()
    pivot.columns.name = None
    year_cols = sorted([c for c in pivot.columns if c not in ("firm", "category")])
    pivot = pivot[["firm", "category"] + year_cols]
    pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)

    print(f"Sentiment scores ({w_sentiment.value}) \u2014 companies as rows, years as columns")
    print(f"Score column: {score_col} | "
          f"Positive \u2265 {w_pos_threshold.value:+.2f} | "
          f"Negative \u2264 {w_neg_threshold.value:+.2f}")
    display(pivot)

    pivot.to_csv("sentiment_scores.csv", index=False)
    print("sentiment_scores.csv saved \u2014 download from the Colab file browser.")

    # LM uncertainty scores (computed automatically when lm_dictionary is selected)
    if "dict_unc" in _df.columns:
        unc_pivot = (_df
                     .pivot_table(index=["firm", "category"],
                                  columns="year",
                                  values="dict_unc",
                                  aggfunc="mean")
                     .round(1)
                     .reset_index())
        unc_pivot.columns.name = None
        year_cols_u = sorted([c for c in unc_pivot.columns if c not in ("firm", "category")])
        unc_pivot = unc_pivot[["firm", "category"] + year_cols_u]
        unc_pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)
        print("\nUncertainty word counts (LM) \u2014 companies as rows, years as columns")
        display(unc_pivot)
        unc_pivot.to_csv("sentiment_scores_uncertainty.csv", index=False)
        print("sentiment_scores_uncertainty.csv saved \u2014 download from the Colab file browser.")

    print()
    print("=" * 65)
    print("  ASSESSMENT SUBMISSION REQUIRED")
    print("  You must include the table(s) above in your written report.")
    print("  Each table must be accompanied by a written discussion that")
    print("  interprets the results with reference to your research question.")
    print("=" * 65)

## Secondary Metric

You must supplement your sentiment analysis with one secondary metric and explain in your report what incremental information it provides that sentiment alone cannot. Three options are scaffolded: the Gunning Fog readability index (Li, 2008), which captures disclosure complexity and has been linked to earnings persistence; year-on-year cosine similarity (Brown and Tucker, 2011), which distinguishes substantive revision from boilerplate repetition; and Latent Dirichlet Allocation, which identifies latent thematic structure without requiring a predefined vocabulary. Your choice should be theoretically motivated by your scenario's research question, with reference to relevant literature, including lecture slides and supporting class activities, core reading and external reading.

**If you select LDA**, you must also choose the number of topics the model should identify. This is a critical methodological decision. Too few topics produces broad, overlapping themes that may not capture meaningful distinctions in the corpus; too many produces highly specific topics that are difficult to interpret and may overfit to noise. There is no universally correct number — it depends on the size and diversity of your corpus and your research question. Common practice is to experiment across a range of values and select the number that produces topics that are both coherent and substantively interpretable. You must justify your chosen number of topics in your report.

**What to do:**
1. **Run Step 7.** A dropdown will appear — select your secondary metric. If you choose LDA, additional inputs will appear for the number of topics and topic labels (fill the labels in after Step 8).
2. **Run Step 8.** The selected metric is computed and applied to the corpus automatically. LDA may take a minute or two to fit. Once Step 8 has finished, **return to Step 7** and fill in the topic labels based on the top words printed in the output.
3. **Run Step 9** to produce the score table for your report.

In [ ]:
#@title Step 7 -- Select your secondary metric

w_secondary = widgets.Dropdown(
    options=["fog_index", "cosine_similarity", "lda"],
    description="Secondary metric:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)

w_lda_topics = widgets.BoundedIntText(
    value=10, min=2, max=50, step=1,
    description="Number of topics:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="380px")
)

# Container for the dynamically generated topic name text boxes
_topic_name_widgets = []
_topic_names_box    = widgets.VBox([])

def _build_topic_name_widgets(_change=None):
    """Rebuild the topic name inputs whenever the topic count changes."""
    global _topic_name_widgets
    n = w_lda_topics.value
    _topic_name_widgets = [
        widgets.Text(
            value=f"Topic {i}",
            description=f"Topic {i} name:",
            style={"description_width": "120px"},
            layout=widgets.Layout(width="400px")
        )
        for i in range(n)
    ]
    _topic_names_box.children = _topic_name_widgets

w_lda_topics.observe(_build_topic_name_widgets, names="value")
_build_topic_name_widgets()   # populate with default 10 topics on load

_lda_panel = widgets.VBox([
    widgets.HTML(value=(
        "<div style='margin-top:10px; padding:12px 14px; background:#1e3a5f; "
        "color:#f8fafc; border-left:4px solid #3b82f6; border-radius:4px; "
        "font-size:13px; line-height:1.6;'>"
        "<b style='font-size:14px;'>LDA settings</b><br><br>"
        "<b>Number of topics:</b> How many distinct themes should the model identify? "
        "Values between 5 and 20 are typical for corpora of this size. Run Step 8, "
        "read the top words printed for each topic, then return here and adjust "
        "the count if the topics are not coherent or meaningful. "
        "You must justify your final choice in your report."
        "<br><br>"
        "<b>Topic names:</b> After running Step 8, read the top words for each "
        "topic printed in the output and give each one a short descriptive label "
        "(e.g. <i>Regulatory Risk</i>, <i>Operational Performance</i>). "
        "These labels replace the raw topic numbers in the Step 9 score table, "
        "making it interpretable. You must discuss what each topic represents "
        "and why it is meaningful for your research question."
        "</div>"
    )),
    w_lda_topics,
    widgets.HTML(value="<b>Topic labels</b> (fill in after running Step 8):"),
    _topic_names_box,
])

_step7_out = widgets.Output()

def _update_step7(_change=None):
    _step7_out.clear_output(wait=True)
    with _step7_out:
        if w_secondary.value == "lda":
            display(_lda_panel)
        print(f"\nSelected: {w_secondary.value}"
              + (f"  |  Topics: {w_lda_topics.value}" if w_secondary.value == "lda" else ""))
        print("Run Step 8 to apply this metric to the corpus.")

w_secondary.observe(_update_step7, names="value")
w_lda_topics.observe(_update_step7, names="value")

display(w_secondary, _step7_out)
_update_step7()

In [ ]:
#@title Step 8 -- Secondary metric

# Install gensim if LDA was selected
if w_secondary.value == "lda":
    import subprocess
    subprocess.run(["pip", "install", "gensim", "-q"], capture_output=True)
    print("gensim installed.")


def gunning_fog(text: str) -> float:
    """
    Gunning Fog Index = 0.4 x (avg sentence length + % complex words).
    Complex words: 3 or more syllables.
    Reference: Li (2008), Journal of Accounting and Economics.
    """
    sentences = [s.strip() for s in re.split(r'[.!?]+', str(text)) if s.strip()]
    if not sentences:
        return np.nan
    words = re.findall(r'\b[a-zA-Z]+\b', str(text))
    if not words:
        return np.nan

    def _syllables(word):
        word = word.lower().rstrip('e')
        return max(1, len(re.findall(r'[aeiouy]+', word)))

    avg_sent_len = len(words) / len(sentences)
    pct_complex  = 100 * sum(1 for w in words if _syllables(w) >= 3) / len(words)
    return round(0.4 * (avg_sent_len + pct_complex), 2)


def cosine_year_on_year(text_year_t: str, text_year_t1: str) -> float:
    """
    TF-IDF cosine similarity between two texts from consecutive years.
    Returns a float in [0, 1]; values near 1 indicate minimal revision.
    Reference: Brown and Tucker (2011), Journal of Accounting Research.
    """
    try:
        vec   = TfidfVectorizer()
        tfidf = vec.fit_transform([str(text_year_t), str(text_year_t1)])
        return round(float(cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]), 4)
    except Exception:
        return np.nan


def run_lda(texts: list, n_topics: int = 10) -> tuple:
    """
    Fit an LDA model to pre-processed texts.
    Returns: (lda_model, topic_word_matrix, document_topic_matrix)
    Reference: Grimmer et al. (2022), Text as Data, Ch. 13.
    """
    from gensim import corpora
    from gensim.models import LdaModel

    tokenised  = [preprocess(t) for t in texts]
    dictionary = corpora.Dictionary(tokenised)
    dictionary.filter_extremes(no_below=5, no_above=0.5)
    bow_corpus = [dictionary.doc2bow(t) for t in tokenised]

    lda_model = LdaModel(
        bow_corpus, num_topics=n_topics,
        id2word=dictionary, passes=10, random_state=STUDENT_SEED
    )

    doc_topics = np.zeros((len(bow_corpus), n_topics))
    for i, bow in enumerate(bow_corpus):
        for tid, prob in lda_model.get_document_topics(bow):
            doc_topics[i, tid] = prob

    return lda_model, lda_model.get_topics(), doc_topics


# --- Apply the selected metric to the corpus ---

_df = corpus

if w_secondary.value == "fog_index":
    print("Computing Gunning Fog Index...")
    corpus["fog"] = corpus["text"].apply(gunning_fog)
    print(f"Done. Mean Fog Index: {corpus['fog'].mean():.2f}")
    display(corpus["fog"].describe().to_frame())

elif w_secondary.value == "cosine_similarity":
    print("Computing year-on-year cosine similarity...")
    _agg = (_df.groupby(["firm", "category", "year"], as_index=False)
               .agg({"text": " ".join}))
    _rows = []
    for _firm, _grp in _agg.groupby("firm"):
        _grp   = _grp.sort_values("year")
        _years = list(_grp["year"])
        _cat   = _grp["category"].iloc[0]
        for i in range(len(_years) - 1):
            t, t1 = _years[i], _years[i + 1]
            txt_t  = _grp[_grp["year"] == t]["text"].values[0]
            txt_t1 = _grp[_grp["year"] == t1]["text"].values[0]
            _rows.append({
                "firm": _firm, "category": _cat,
                "year_pair": f"{t}-{t1}",
                "similarity": cosine_year_on_year(txt_t, txt_t1)
            })
    similarity_df = pd.DataFrame(_rows)
    print(f"Done. {len(similarity_df)} firm-year pairs computed.")
    display(similarity_df)

elif w_secondary.value == "lda":
    _n_topics = w_lda_topics.value
    print(f"Fitting LDA model ({_n_topics} topics) ...")
    lda_model, topic_word_matrix, document_topic_matrix = run_lda(
        list(corpus["text"].fillna("")), n_topics=_n_topics
    )
    corpus["dominant_topic"] = document_topic_matrix.argmax(axis=1)
    print(f"Done. Top words per topic ({_n_topics} topics):")
    for _i, _topic in enumerate(lda_model.print_topics(num_words=8)):
        print(f"  Topic {_i}: {_topic[1]}")

In [ ]:
#@title Step 9 -- Secondary metric score table

# Run this cell after Step 8 has completed.
# It pivots the secondary metric scores into a company x year table
# ready to copy into your written report.

_df = corpus

_SUBMISSION_REMINDER = (
    "\n" + "=" * 65 + "\n"
    "  ASSESSMENT SUBMISSION REQUIRED\n"
    "  You must include the table above in your written report.\n"
    "  Discuss what the scores reveal about your research question.\n"
    + "=" * 65
)

if w_secondary.value == "fog_index":

    if "fog" not in _df.columns:
        print("Fog scores not found. Run Step 8 first.")
    else:
        fog_pivot = (_df
                     .pivot_table(index=["firm", "category"],
                                  columns="year",
                                  values="fog",
                                  aggfunc="mean")
                     .round(2)
                     .reset_index())
        fog_pivot.columns.name = None
        year_cols = sorted([c for c in fog_pivot.columns if c not in ("firm", "category")])
        fog_pivot = fog_pivot[["firm", "category"] + year_cols]
        fog_pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)
        print("Gunning Fog scores \u2014 companies as rows, years as columns")
        display(fog_pivot)
        fog_pivot.to_csv("secondary_metric_fog.csv", index=False)
        print("secondary_metric_fog.csv saved \u2014 download from the Colab file browser.")
        print(_SUBMISSION_REMINDER)

elif w_secondary.value == "cosine_similarity":

    try:
        sim_pivot = (similarity_df
                     .pivot_table(index=["firm", "category"],
                                  columns="year_pair",
                                  values="similarity",
                                  aggfunc="mean")
                     .round(4)
                     .reset_index())
        sim_pivot.columns.name = None
        sim_pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)
        print("Year-on-year cosine similarity \u2014 companies as rows, year pairs as columns")
        display(sim_pivot)
        sim_pivot.to_csv("secondary_metric_cosine.csv", index=False)
        print("secondary_metric_cosine.csv saved \u2014 download from the Colab file browser.")
        print(_SUBMISSION_REMINDER)
    except NameError:
        print("similarity_df not found. Run Step 8 first.")

elif w_secondary.value == "lda":

    if "dominant_topic" not in _df.columns:
        print("LDA results not found. Run Step 8 first.")
    else:
        _topic_labels = {
            i: (w.value.strip() or f"Topic {i}")
            for i, w in enumerate(_topic_name_widgets)
        }

        lda_pivot = (_df
                     .pivot_table(index=["firm", "category"],
                                  columns="year",
                                  values="dominant_topic",
                                  aggfunc=lambda x: int(x.mode().iloc[0]) if not x.empty else -1)
                     .reset_index())
        lda_pivot.columns.name = None
        year_cols = sorted([c for c in lda_pivot.columns if c not in ("firm", "category")])

        for col in year_cols:
            lda_pivot[col] = lda_pivot[col].map(
                lambda v: _topic_labels.get(v, f"Topic {v}") if v != -1 else ""
            )

        lda_pivot = lda_pivot[["firm", "category"] + year_cols]
        lda_pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)
        print("Dominant LDA topic \u2014 companies as rows, years as columns")
        print("Topic numbers have been replaced with your labels from Step 7.")
        display(lda_pivot)
        lda_pivot.to_csv("secondary_metric_lda.csv", index=False)
        print("secondary_metric_lda.csv saved \u2014 download from the Colab file browser.")
        print(_SUBMISSION_REMINDER)

print("\n" + "=" * 65)
print("  REPORT WRITING PROMPT")
print("  Compare the values above against your sentiment score table")
print("  from Step 6e for the same firms and years.")
print("  Find ONE firm-year where the two metrics tell different stories.")
print("  Example: a firm scoring positively on sentiment but with an")
print("  unusually high Fog Index — or stable sentiment alongside a sharp")
print("  drop in year-on-year cosine similarity.")
print("  Record the firm name, year, and both numerical values.")
print("  This is the material for your Analytical Contradiction (Output 8).")
print("=" * 65)


## Visualisation

Step 10 generates two standard figures automatically:

1. **Sentiment over time** — a line chart of mean sentiment score by year, broken down by firm category, and a box plot of the score distribution per category.
2. **Secondary metric over time** — equivalent charts for whichever secondary metric you selected in Step 7.

Both figures are saved to disk as PNG files so they can be inserted directly into your written report. Your report must include both figures, provide fully labelled captions, and discuss what each figure reveals about your research question.

**What to do:** Run Step 10. Both figures will be generated and saved automatically. To download them, open the Colab file browser (the folder icon in the left-hand panel), locate `fig1_sentiment.png` and `fig2_*.png`, right-click each file, and select **Download**.

In [ ]:
#@title Step 10 -- Visualisations

_df = corpus

_FIG_REMINDER = (
    "=" * 65 + "\n"
    "  ASSESSMENT SUBMISSION REQUIRED\n"
    "  You must include this chart in your written report.\n"
    "  Provide a fully labelled caption and a written discussion\n"
    "  that interprets the figure with reference to your research question.\n"
    + "=" * 65
)

# --- Figure 1: Sentiment ---

_score_col = ("dict_score" if "dict_score" in _df.columns else
              "ml_score"   if "ml_score"   in _df.columns else None)

if _score_col is None:
    print("No sentiment scores found. Run Step 6a or Step 6b first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    yearly = _df.groupby(["year", "category"])[_score_col].mean().reset_index()
    for cat, grp in yearly.groupby("category"):
        axes[0].plot(grp["year"], grp[_score_col], marker="o", label=cat)
    axes[0].axhline(0, color="grey", linestyle="--", linewidth=0.8)
    axes[0].axhline(w_pos_threshold.value, color="#22c55e", linestyle=":", linewidth=0.8,
                    label=f"Pos threshold ({w_pos_threshold.value:+.2f})")
    axes[0].axhline(w_neg_threshold.value, color="#ef4444", linestyle=":", linewidth=0.8,
                    label=f"Neg threshold ({w_neg_threshold.value:+.2f})")
    axes[0].set_title(f"Mean sentiment score by year\n({w_sentiment.value})")
    axes[0].set_xlabel("Year")
    axes[0].set_ylabel("Mean sentiment score")
    axes[0].legend(title="Category", fontsize=8)

    _cats = sorted(_df["category"].unique())
    axes[1].boxplot(
        [_df[_df["category"] == c][_score_col].dropna().values for c in _cats],
        labels=_cats, patch_artist=True
    )
    axes[1].axhline(0, color="grey", linestyle="--", linewidth=0.8)
    axes[1].axhline(w_pos_threshold.value, color="#22c55e", linestyle=":", linewidth=0.8)
    axes[1].axhline(w_neg_threshold.value, color="#ef4444", linestyle=":", linewidth=0.8)
    axes[1].set_title(f"Sentiment distribution by category\n({w_sentiment.value})")
    axes[1].set_xlabel("Category")
    axes[1].set_ylabel("Sentiment score")
    plt.setp(axes[1].get_xticklabels(), rotation=15)

    plt.tight_layout()
    plt.savefig("fig1_sentiment.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("fig1_sentiment.png saved.")
    print(_FIG_REMINDER)


# --- Figure 2: Secondary metric ---

if "fog" in _df.columns:

    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
    yearly2 = _df.groupby(["year", "category"])["fog"].mean().reset_index()
    for cat, grp in yearly2.groupby("category"):
        axes2[0].plot(grp["year"], grp["fog"], marker="o", label=cat)
    axes2[0].set_title("Mean Gunning Fog Index by year")
    axes2[0].set_xlabel("Year")
    axes2[0].set_ylabel("Gunning Fog Index")
    axes2[0].legend(title="Category", fontsize=8)

    _cats2 = sorted(_df["category"].unique())
    axes2[1].boxplot(
        [_df[_df["category"] == c]["fog"].dropna().values for c in _cats2],
        labels=_cats2, patch_artist=True
    )
    axes2[1].set_title("Gunning Fog distribution by category")
    axes2[1].set_xlabel("Category")
    axes2[1].set_ylabel("Gunning Fog Index")
    plt.setp(axes2[1].get_xticklabels(), rotation=15)
    plt.tight_layout()
    plt.savefig("fig2_fog.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("fig2_fog.png saved.")
    print(_FIG_REMINDER)

elif "dominant_topic" in _df.columns:

    fig2, ax2 = plt.subplots(figsize=(10, 5))
    _tc = _df.groupby(["category", "dominant_topic"]).size().unstack(fill_value=0)
    _tc.div(_tc.sum(axis=1), axis=0).plot(
        kind="bar", stacked=True, ax=ax2, colormap="tab20"
    )
    ax2.set_title("Dominant LDA topic distribution by category")
    ax2.set_xlabel("Category")
    ax2.set_ylabel("Proportion of documents")
    ax2.legend(title="Topic", fontsize=7, bbox_to_anchor=(1.05, 1))
    plt.setp(ax2.get_xticklabels(), rotation=15)
    plt.tight_layout()
    plt.savefig("fig2_lda.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("fig2_lda.png saved.")
    print(_FIG_REMINDER)

elif "similarity_df" in dir():

    try:
        fig2, ax2 = plt.subplots(figsize=(10, 5))
        for cat, grp in similarity_df.groupby("category"):
            sim_mean = grp.groupby("year_pair")["similarity"].mean()
            ax2.plot(sim_mean.index, sim_mean.values, marker="o", label=cat)
        ax2.set_title("Year-on-year cosine similarity by category")
        ax2.set_xlabel("Year pair")
        ax2.set_ylabel("Cosine similarity (0 = no overlap, 1 = identical)")
        ax2.legend(title="Category", fontsize=8)
        plt.setp(ax2.get_xticklabels(), rotation=15)
        plt.tight_layout()
        plt.savefig("fig2_cosine.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("fig2_cosine.png saved.")
        print(_FIG_REMINDER)
    except Exception as _e:
        print(f"Could not plot cosine similarity: {_e}")

else:
    print("No secondary metric scores found. Run Step 8 first.")

## Manual Validation

Automated sentiment methods require empirical validation. You must draw a random sample of 50 sentences from the corpus, classify each sentence as positive (1), neutral (0), or negative (−1) by reading it in context, and compare your human judgements to the automated scores.

**Your sample is unique to you.** The sentences are drawn using your personal seed (derived from your student number), so no two students receive the same 50 sentences.

**The sample is stratified** to ensure at least 12 sentences from each sentiment class (Positive, Neutral, Negative). This guarantees that the validation exercise covers all three labels, even if your corpus is skewed towards one class.

**What to do with the downloaded CSV:**
1. Open `manual_validation_sample.csv` in Excel or Google Sheets.
2. Read each sentence in the `sentence_text` column carefully.
3. Enter your own label in the `human_label` column: 1, 0, or −1.
4. Look for sentences where your label **disagrees** with `automated_label`. For those sentences, note the specific word or phrase that likely caused the misclassification — this is the material for your sentence-level analysis in the report.

**Report requirement:** Identify three sentences where your label differs from the automated label. For each, quote the sentence verbatim, state both labels, identify the misclassifying word(s), and suggest a specific correction. See Output 6 in the brief.


In [ ]:
#@title Step 11 -- Draw stratified random sample for manual validation

# Uses STUDENT_SEED for reproducibility — your 50 sentences are unique to you.

import random as _random

_rng = _random.Random(STUDENT_SEED)

# Determine automated score column
_auto_col = ('dict_score' if 'dict_score' in corpus.columns else
             'ml_score'   if 'ml_score'   in corpus.columns else None)

if _auto_col is None:
    print('ERROR: No sentiment scores found. Run Step 6a/6b/6c/6d first.')
else:
    _pos_t = w_pos_threshold.value
    _neg_t = w_neg_threshold.value

    # Sentence-level tokenisation
    _sent_rows = []
    for _, doc_row in corpus.iterrows():
        for _sent in sent_tokenize(str(doc_row['text'])):
            _sent = _sent.strip()
            if len(_sent.split()) < 5:
                continue
            _score = float(doc_row[_auto_col])
            _auto_lbl = 1 if _score >= _pos_t else (-1 if _score <= _neg_t else 0)
            _sent_rows.append({
                'firm':           doc_row['firm'],
                'year':           doc_row['year'],
                'section':        doc_row.get('section',''),
                'sentence_text':  _sent,
                'auto_score':     round(_score, 4),
                'automated_label': _auto_lbl,
                'human_label':    '',
            })

    _all = pd.DataFrame(_sent_rows)

    # Stratified sampling: at least 12 per class, 14 random remainder
    _N_PER_CLASS = 12
    _N_RANDOM    = 14
    _sampled = []
    for _lbl in [1, 0, -1]:
        _pool = _all[_all['automated_label'] == _lbl]
        _n = min(_N_PER_CLASS, len(_pool))
        if _n > 0:
            _lbl_seed = STUDENT_SEED + {1: 7, 0: 13, -1: 19}[_lbl]  # positive offsets only
        _sampled.append(_pool.sample(n=_n, random_state=_lbl_seed))
    _sampled_orig_idx = set()
    for _chunk in _sampled:
        _sampled_orig_idx.update(_chunk.index.tolist())
    _sampled_df = pd.concat(_sampled, ignore_index=True)

    # Fill remainder from unstratified pool
    _remaining_needed = 50 - len(_sampled_df)
    if _remaining_needed > 0:
        _rest = _all[~_all.index.isin(_sampled_orig_idx)]
        _n_fill = min(_remaining_needed, len(_rest))
        if _n_fill > 0:
            _fill = _rest.sample(n=_n_fill, random_state=STUDENT_SEED + 99)
            _sampled_df = pd.concat([_sampled_df, _fill], ignore_index=True)

    _sampled_df = _sampled_df.sample(frac=1, random_state=STUDENT_SEED).reset_index(drop=True)
    _sampled_df.insert(0, 'sentence_id', range(1, len(_sampled_df)+1))

    manual_validation_sample = _sampled_df[['sentence_id','firm','year','section',
                                            'sentence_text','automated_label','human_label']]

    # Display all 50 rows — do not truncate
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', 120)
    from IPython.display import display, HTML
    display(HTML(manual_validation_sample.to_html(index=False)))

    # Download
    manual_validation_sample.to_csv('manual_validation_sample.csv', index=False)
    print('\nFile saved: manual_validation_sample.csv')
    _dist = manual_validation_sample['automated_label'].value_counts().to_dict()
    print(f'Class distribution — Positive: {_dist.get(1,0)}, '
          f'Neutral: {_dist.get(0,0)}, Negative: {_dist.get(-1,0)}')
    print('\n' + '=' * 65)
    print('  REPORT WRITING PROMPT')
    print('  Open manual_validation_sample.csv and read each sentence.')
    print('  Fill in the human_label column (1 / 0 / -1) for all 50 rows.')
    print('  Then identify THREE sentences where your label differs from')
    print('  automated_label. For each:')
    print('    1. Quote the sentence verbatim')
    print('    2. State both labels (automated vs. human)')
    print('    3. Identify the specific word(s) causing the mismatch')
    print('    4. Suggest a concrete fix (e.g. "charged" is context-dependent)')
    print('  This is the material for Output 6 in your written report.')
    print('=' * 65)


## Analytical Contradiction

Before submitting, you are required to produce one additional piece of analysis that directly cross-references your two main outputs.

**What to do:** Using the pivot tables from Step 6e (sentiment scores) and Step 9 (secondary metric scores), identify **one firm-year** where the two metrics tell contradictory stories. Examples:

- A firm with a **positive sentiment score** in a year when its **Fog Index is unusually high** (positive messaging accompanied by complex, potentially obfuscatory language)
- A firm with a **stable or rising sentiment score** alongside a **sharp drop in year-on-year cosine similarity** (the language changed dramatically despite positive tone)
- A firm where the **automated label is positive** but known sector events in that year would predict otherwise (the model is failing on this firm-year)

**In your report:** State the firm name, the year, and the specific numerical values from both tables. Explain the contradiction and what it reveals about the limitations of relying on a single metric. This is **Output 8** in your written report and is a required section — it will not attract marks if it is absent or if it does not cite specific numerical values from your own outputs.

## Submit and Summary

**What to do:**
1. **Run Step 12** once you have finalised all your choices above. This submits your selections to the module log. You can re-run it if you change a choice — each submission is recorded separately.
2. **Run Step 13** to generate a formatted summary table of all your methodological decisions. Copy this table into your reflective document.

In [ ]:
#@title Step 12 -- Submit your methodological choices

# Run this cell after finalising all choices above.
# You may re-run it if you change a choice; each submission is
# recorded separately and the submission_count column will increment.
# Note the submission number printed below -- it is included in the
# Step 13 summary table so you can match your summary to the correct
# row in the module log.

_submission_number = None   # populated on successful submission


def submit_choices(apps_script_url, student_id, scenario_id):
    global _submission_number

    if not apps_script_url:
        print("APPS_SCRIPT_URL is not set. Choices not submitted.")
        print("Contact your module leader if you are unsure how to proceed.")
        return

    payload = {
        "student_id":           student_id,
        "scenario":             scenario_id,
        "corpus_section":       w_section.value,
        "case_folding":         w_case_folding.value,
        "stopword_list":        w_stopwords.value,
        "normalisation_method": w_normalisation.value,
        "number_handling":      w_numbers.value,
        "tfidf_weighting":      w_tfidf.value,
        "sentiment_model":      w_sentiment.value,
        "pos_threshold":        w_pos_threshold.value,
        "neg_threshold":        w_neg_threshold.value,
        "secondary_metric":     w_secondary.value,
    }

    # Include topic count for LDA
    if w_secondary.value == "lda":
        payload["lda_n_topics"] = w_lda_topics.value

    try:
        response = requests.post(
            apps_script_url,
            data=json.dumps(payload),
            headers={"Content-Type": "application/json"},
            timeout=15
        )
        result = response.json()
        if result.get("status") == "success":
            _submission_number = result.get("submission")
            print(f"Choices submitted successfully.")
            print(f"  Student ID:        {student_id}")
            print(f"  Submission number: {_submission_number}")
            print()
            print("Record this submission number. It is included in the Step 13")
            print("summary table so the module team can locate your entry in the log.")
        else:
            print(f"Submission returned an error: "
                  f"{result.get('message', 'unknown error')}")
    except requests.exceptions.Timeout:
        print("The request timed out. Check your internet connection and try again.")
    except Exception as ex:
        print(f"Submission failed: {ex}")


submit_choices(APPS_SCRIPT_URL, STUDENT_ID, SCENARIO["id"])

In [ ]:
#@title Step 13 -- Your choices summary

# This cell generates a formatted table of all your methodological choices.
# Copy it into your reflective document.

_choices = [
    ("Submission number",
     str(_submission_number) if _submission_number is not None
     else "Not yet submitted \u2014 run Step 12 first"),
    ("Student ID",        STUDENT_ID),
    ("Scenario",          f"{SCENARIO['id']} \u2014 {SCENARIO['title']}"),
    ("Corpus section",    w_section.value),
    ("Case folding",      w_case_folding.value),
    ("Stop-word list",    w_stopwords.value),
    ("Normalisation",     w_normalisation.value),
    ("Number handling",   w_numbers.value),
    ("TF-IDF weighting",  w_tfidf.value),
    ("Model family",     "Transformer (FinBERT / DistilBERT)" if w_sentiment.value in ("finbert","distilbert") else "Classical (dictionary or ML)"),
    ("Sentiment model",   w_sentiment.value),
    ("Positive threshold (\u2265)", f"{w_pos_threshold.value:+.2f}"),
    ("Negative threshold (\u2264)", f"{w_neg_threshold.value:+.2f}"),
    ("Secondary metric",  w_secondary.value),
]

# LDA choices
if w_secondary.value == "lda":
    _choices.append(("LDA number of topics", str(w_lda_topics.value)))
    _topic_label_str = "; ".join(
        f"{i}: {w.value.strip() or f'Topic {i}'}"
        for i, w in enumerate(_topic_name_widgets)
    )
    _choices.append(("LDA topic labels", _topic_label_str))

summary_df = pd.DataFrame(_choices, columns=["Choice", "Selection"]).set_index("Choice")
print("Methodological choices \u2014 copy this table into your reflective document:\n")
display(summary_df)